# Mulligan Analysis
Our goal is to analyze different mulligan rules from various perspectives. We will compare different mulligan rules by looking at probabilistic measures of how well different strategies perform with them.


## Intentions
The ultimate goal of this document is to do some computations in a way that people understand. I apologize in advance if it seems overly technical and will be happy to add further explanation to various parts. This is not my area of expertise, so I probably don't have the clearest explanations. So please get in touch to let me know how I can improve this. Especially if you notice a mistake!!

## Initial thoughts
We collect a few general thoughts below.
### The Context
The context of this is a game called Hubworld: Aidalon. The goal in this game is to capture an opponents 'agents' before they capture your agents. As this is the win condition, it is often preferable to not have agents in your opening hand. It is also the case that other cards are necessary for defending as well as for generating income. Having these in an opening hand is ideal. Other strategies, 'mulliganing hard' for core pieces of your deck are also viable.

Our goal is not to determine what the best strategy is but to determine which mulligans best support certain outcomes. We will try to be agnostic and explain the relevant mathematics when it arises.

### Expected Value and Probability
The most familiar way of measuring an undesireable outcome is to state the probability of it ovvuring, or of it not occuring. However, there may be another more relevant probabilistic measure: Expected Value. In this case, we could ask what the expected value is of the number of agents in an opening hand of 5 cards is, without discarding any cards. This would be computed as follows. The expected value of the number of agents would be computed as:

$$E[\text{Number of Agents}]= \Sigma_{k=0}^5 P(\text{k Agents})\cdot k$$

It is a weighted sum that can be very useful. We will compute this when comparing different mulligan systems as well as the standard probabilities.

### Spectrum of Strategies
There are many different strategies that one can pursue. We currently only consider the following:
- Minimize Agents in opening hand.
- Hard mulligan for desired cards.
- - This strategy includes maximizing Agents in an opening hand.

### General approach
We will approach the problem by dividing the mulligan process into different steps. This will allow us to use the total probability law which states that the probability of an event is the sum of conditional probabilities weighted by the probability of the condition occurring. This will often make our computations easier to approach. We will also do expected value computations. After doing this for different mulligans we will compare the results of the computations.

### Different Mulligans
Here we outline the different mulligan rules we will be considering.

#### The 'Friendliest' Mulligan
This is the current mulligan system. It is the one found in Earthborne Rangers (and in Arkham Horror the Card Game, albeit modified). The player draws an initial hand of $5$ cards. They set aside any number of cards from this initial draw and then redraw to  back up to a total of $5$ cards. The set aside cards are then shuffled back into the deck. The downside of this is that it is possible to set aside cards and then draw into undesired cards. This makes a good computation difficult as there may be an exploding tree of situations to condition your computation on.

#### The '7/5' Mulligan
This is the proposed mulligan. It is simple and straight forward. The player draws an initial hand of $7$ cards. They then discard $2$ of them and shuffle them back into the deck. The upside of this is that players see more cards at the outset. It is also not possible to draw into undesirable cards. 

### Relevant constants
In the game in question, there are $36$ cards to a deck and $6$ of them are agents. There are other types of cards, economic, defensive, and offensive cards. The rest of the composition of the deck is up to the player (aside from adherence to rules that do not impact the distribution of such cards).

We will use `total` to denote the total number of cards in the deck, `marked` to denote the number of marked cards in the deck, `initial_draw` to denote number of cards drawn initially, `hand_size` to denote the size of the final hand. However, when they agree we will just use `hand_size`. The variables $k$ and $m$ will be reserved for the number of marked cards in the initial draw and final hand, respectively.

**Note:** Agents can be thought of as marked cards, these are just any card in a subset of a certain size. We may also use the agent terminology when discussing the problem as this is the example at the forefront of the analysis. We will also use the term 'initial draw' to refer to the first group of cards drawn from the deck before the mulligan, while we will use opening hand to refer to the hand of cards the player begins the game with, and final hand to refer to the hand the player has after the mulligan. Opening hand and final hand are in essence the same but slightly different semantically.

Below are the default parameters. They can be changed and the computations can all be rerun to see how these parameters impact the computation. Default values are given in comment after each assignment.

In [176]:
total = 36 # 36 total cards in the deck

marked = 6 # 6 agents in the deck
initial_draw = 5 # 5 cards in the initial hand
hand_size = 5 # 5 cards in the hand after mulligan

We will often use default values when explaining computations.

### Formatting Functions
Here are some simple functions that we will use for formatting. To change the number of decimal places displayed just change the value of `percentage_decimals` or `decimals` and execute the cell.

In [177]:
# Set up formatting functions

percentage_decimals = 4
def format_percent(x):
    return f"{x*100:.{percentage_decimals}f}%"

decimals=3
def format_decimals(x):
    return f"{x:.{decimals}f}"

# Minimizing Agents
We first consider the problem of minimizing the number of agents in an opening hand. The strategy for this is to discard a card if and only if it is an agent. More generally, you can consider the agents to be a particular set of undesirable cards.

The computation for maximizing agents is a a consequence of the strategy to mulligan for desired cards and so we leave it as a corollary.

## The 'Friendliest Mulligan'

### The Initial Draw
We will now do some baseline computations for the initial draw to see what the distribution of agents is among the initial cards drawn. We will also compute the expected number of agents is in the initial draw.

#### $P(\text{k Agents})$
The probability of drawing $k$ agents from a standard deck among $5$ cards is given by the formula

$$P(\text{k Agents})=\frac{\binom{6}{k}\binom{31}{5-k}}{\binom{36}{5}}$$

or abstractly

$$P(\text{k Agents})=\frac{\binom{marked}{k}\binom{total-marked}{initial\_ draw-k}}{\binom{total}{hand\_ size}}.$$

We can use this to compute the probabilities of getting $k$ agents in our initial draw.

In [178]:
import numpy as np
import pandas as pd
from scipy.special import comb

# Calculate probabilities for k agents (k = 0 to 5)

def compute_initial_probability(k, marked, initial_draw, total):
    num = comb(marked, k, exact=True) * comb(total - marked, initial_draw - k, exact=True)
    denom = comb(total, initial_draw, exact=True)
    prob = num / denom
    return prob

def compute_all_initial_probabilities(marked, initial_draw, total):
    k_values = range(0, min(marked, initial_draw) + 1)  # k can only be from 0 to min(n, h_0)
    probabilities = {}
    for k in k_values:
        probabilities[k] = compute_initial_probability(k, marked, initial_draw, total)
    return probabilities


def build_initial_probability_table(marked, initial_draw, total):
    k_values = range(0, min(marked, initial_draw) + 1)  # k can only be from 0 to min(n, h_0)
    probabilities = {}

    probabilities = compute_all_initial_probabilities(marked, initial_draw, total)
    # Create DataFrame
    df = pd.DataFrame({
        'k (Agents)': k_values,
        'Probability': [f"{format_percent(p)}" for p in probabilities.values()]
    })
    return df
percentage_decimals = 2
df = build_initial_probability_table(marked, initial_draw, total)
df

,k (Agents),Probability
0,0,37.80%
1,1,43.62%
2,2,16.15%
3,3,2.31%
4,4,0.12%
5,5,0.00%


#### Expected Number of Agents in the Initial Draw
We will stick to using the default values when explaining things as it will be more relevant that way for the intended audience.

$$E[\text{Number of Agents}]=\Sigma_{k=0}^{5}k\cdot P(\text{k Agents})$$

We now compute the expected number of agents in the initial draw:

In [179]:
decimals = 7
def compute_n_display_expected_value(probabilities, range_limit):
    expected_value = 0
    for k in range(0, range_limit + 1):
        contribution = probabilities[k] * k
        expected_value += contribution
        print(f"k = {k}: P({k}) × {k} = {format_decimals(probabilities[k])} × {k} = {contribution:.6f}")
    return expected_value
probabilities = compute_all_initial_probabilities(marked, initial_draw, total)
expected_value = compute_n_display_expected_value(probabilities, min(marked, initial_draw))
decimals = 3
print(f"Expected Value = {format_decimals(expected_value)} agents")

k = 0: P(0) × 0 = 0.3780080 × 0 = 0.000000
k = 1: P(1) × 1 = 0.4361631 × 1 = 0.436163
k = 2: P(2) × 2 = 0.1615419 × 2 = 0.323084
k = 3: P(3) × 3 = 0.0230774 × 3 = 0.069232
k = 4: P(4) × 4 = 0.0011937 × 4 = 0.004775
k = 5: P(5) × 5 = 0.0000159 × 5 = 0.000080
Expected Value = 0.833 agents


Thus the expected number of agents is  $0.833=\frac{5}{6}$. This means that in the initial draw you can expect to see 1 agent.

### After the Mulligan
We will now look at what the distribution of agents looks like at the end of the mulligan process. We will do this by breaking up the situation into different cases and computing conditional probabilities and expected values. We will compute the marginal probabilities using the law of total probabilities. This is the formula:

$$ P(A) = \Sigma_i P(A | B_i) \cdot P(B_i) $$

where the $B_i$ partition the sample space. We will be partitioning the sample space based on the number of agents in the initial draw.

It would be possible to do the computation by computing the joint distribution directly, but we feel this approach makes a lot of sense conceptually.

#### $P(\text{m Agents} | \text{k Agents})$
<!-- VOICE CHECK: Shifts from "we will compute" to "where m4 is" (objective statement). Also note: "m4" appears to be typo for "m" -->
We will compute $P(\text{m Agents} | \text{k Agents})$ for various $m$ and $k$. $m$ is the number of agents in the hand after the mulligan and $k$ is the number of agents in the initial draw. Again, this is assuming the strategy is to discard a card if and only if it is an agents.
Recall that the definition of conditional probability leads to

$$ P(\text{m Agents}|\text{k Agents})= \frac{P(\text{m Agents}\cap\text{k Agents})}{P(\text{k Agents})}.$$

In our case, we arrive at:

$$ P(\text{m Agents} | \text{k Agents})= \frac{\binom{marked-k}{m} \cdot \binom{total-hand\_ size-(marked-k)}{k-m}}{\binom{total- hand\_ size}{k}}=\frac{\binom{6-k}{m} \cdot \binom{31-(6-k)}{k-m}}{\binom{31}{k}}.$$

This can be explained as follows. You draw $5$ cards in the initial draw, $k$ of them are agents. There are now $31$ remaining cards in the deck and $6-k$ of them are agents. So there are $\binom{31-(6-k)}{k-m}$ ways of choosing non-agents and $\binom{6-k}{m}$ ways of choosing agents when redrawing $k$ cards out of a total of $\binom{31}{k}$ total ways of redrawing $k$ cards.

In [180]:
from scipy.special import comb

# Calculate P(m | k) for all k and m from 0 to 5

def compute_pmk_details(m,k, marked, total, hand_size):
    # Calculate intermediate values
    remaining_marked = marked - k
    remaining_non_marked = (total-hand_size)-(marked-k)
    num_a = 0 if m > remaining_marked else comb(remaining_marked, m, exact=True)
    num_b = 0 if k - m > remaining_non_marked else comb(remaining_non_marked, k - m, exact=True)
    num = 0 if m > k else num_a * num_b
    denom = comb(31, k, exact=True)
    
    if denom > 0:
        prob_conditional = num / denom
    else:
        prob_conditional = 0.0
    p_k = compute_initial_probability(k, marked, initial_draw, total)
    prob_joint = prob_conditional * p_k
    details = {
                'num_a': num_a,
                'num_b': num_b,
                'num': num,
                'denom': denom,
                'p_conditional': prob_conditional,
                'p_joint': prob_joint,
                'p_k': p_k
            }
    
    return details

# k is the initial key as the computation for m depends on it.
def conditional_calculation_details(marked, total, hand_size):
    calculation_details = {}
    for m in range(hand_size + 1):
        calculation_details[m] = {}
        for k in range(hand_size + 1):
            calculation_details[m][k] = compute_pmk_details(m, k, marked, total, hand_size)
    return calculation_details
            
calculation_details = conditional_calculation_details(marked, total, hand_size)
percentage_decimals = 2
def build_conditional_probabilities(calculation_details):
    
    # Create a comprehensive DataFrame with all pairs (k, m) and all calculation details
    data = []
    
    for m in calculation_details.keys():
        for k in calculation_details[m].keys():
            details = calculation_details[m][k]
            data.append({
                'm': m,
                'k': k,
                'num_a': details['num_a'],
                'num_b': details['num_b'],
                'num': details['num'],
                'denom': details['denom'],
                #'P(m|k)': details['p_conditional'],
                #'P(k)': details['p_k'],
                #'P(m and k)': details['p_joint'],
                'P(m|k) %': format_percent(details['p_conditional']),
                'P(k) %': format_percent(details['p_k']),
                'P(m and k) %': format_percent(details['p_joint'])
            })

    return pd.DataFrame(data)



We now want to display the results. For reasons of space we will omit all rows where the value of the num is 0 (and so the resulting probability will be 0 as well).

In [181]:
df_all = build_conditional_probabilities(calculation_details)
df_display = df_all[df_all['num'] != 0]
print("Conditional Probability Table with Calculation Details")
print("=" * 150)
print("Columns in dataframe: m (final agents), k (initial agents), num_a, num_b, num, denom,")
print(22*' ' + "P(m|k) %, P(k) %, P(m and k) %")
print("=" * 150)
print(df_display.to_string(index=False))
print("=" * 150)


Conditional Probability Table with Calculation Details
Columns in dataframe: m (final agents), k (initial agents), num_a, num_b, num, denom,
                      P(m|k) %, P(k) %, P(m and k) %
 m  k  num_a  num_b    num  denom P(m|k) % P(k) % P(m and k) %
 0  0      1      1      1      1  100.00% 37.80%       37.80%
 0  1      1     26     26     31   83.87% 43.62%       36.58%
 0  2      1    351    351    465   75.48% 16.15%       12.19%
 0  3      1   3276   3276   4495   72.88%  2.31%        1.68%
 0  4      1  23751  23751  31465   75.48%  0.12%        0.09%
 0  5      1 142506 142506 169911   83.87%  0.00%        0.00%
 1  1      5      1      5     31   16.13% 43.62%        7.03%
 1  2      4     27    108    465   23.23% 16.15%        3.75%
 1  3      3    378   1134   4495   25.23%  2.31%        0.58%
 1  4      2   3654   7308  31465   23.23%  0.12%        0.03%
 1  5      1  27405  27405 169911   16.13%  0.00%        0.00%
 2  2      6      1      6    465    1.29% 16.15% 

<!-- VOICE CHECK: Shifts perspective from "we" voice to imperative "enter values" and "execute cells" (direct reader instruction) -->
To zoom in on particular cases, enter values in the list for the number of final agents you want to consider and then execute the following cells. If the situation is impossible "N/A (invalid)" will be listed. If $k$ is zero then "redraw" will be listed.

In [182]:
possible_m_values = [0,1,2]
print(f"Calculations for m = {possible_m_values} agents in final hand:")

Calculations for m = [0, 1, 2] agents in final hand:


In [183]:

# Also display tables for each fixed m
print("\n\nDetailed View by Fixed m (Final Agent Count):")
print("=" * 80)


def build_conditional_probabilities_by_m(calculation_details, m):
    print(f"\nFor m = {m} agents (final hand):")
    print("-" * 80)
    data = []
    for k in calculation_details[m].keys():
        details = calculation_details[k][m]
        
        if k == 0:
            calc_str = "No redraw"
        elif details['num'] == 0:
            calc_str = "N/A (invalid)"
        else:
            calc_str = f"C({6-k},{m}) × C({25+k},{k-m}) / C(31,{k})"
        
        data.append({
            'k (Initial Agents)': k,
            'Calculation': calc_str,
            'P(m|k)': f"{details['p_conditional']:.6f}",
            'P(k)': f"{details['p_k']:.6f}",
            'P(m and k)': f"{details['p_joint']:.6f}",
            'Percentage': f"{details['p_conditional']*100:.2f}%"
        })
    return pd.DataFrame(data)
    
for m in possible_m_values:
    print(f"\nFor m = {m} agents (final hand):")
    print("-" * 80)
    
    df_m = build_conditional_probabilities_by_m(calculation_details, m)
    print(df_m.to_string(index=False))

print("\n" + "=" * 80)



Detailed View by Fixed m (Final Agent Count):

For m = 0 agents (final hand):
--------------------------------------------------------------------------------

For m = 0 agents (final hand):
--------------------------------------------------------------------------------
 k (Initial Agents)   Calculation   P(m|k)     P(k) P(m and k) Percentage
                  0     No redraw 1.000000 0.378008   0.378008    100.00%
                  1 N/A (invalid) 0.000000 0.378008   0.000000      0.00%
                  2 N/A (invalid) 0.000000 0.378008   0.000000      0.00%
                  3 N/A (invalid) 0.000000 0.378008   0.000000      0.00%
                  4 N/A (invalid) 0.000000 0.378008   0.000000      0.00%
                  5 N/A (invalid) 0.000000 0.378008   0.000000      0.00%

For m = 1 agents (final hand):
--------------------------------------------------------------------------------

For m = 1 agents (final hand):
---------------------------------------------------------------

Now we are in a position to compute the probability of having $m$ agents after the mulligan. We will use that conditioning on $k$ makes each scenario distinct and so we are able to use additivity of disjoint events. The inclusion-exclusion principle says that 

$$ P(A \cup B) = P(A) + P(B) - P(A \cap B).$$

In english, this reads as "the probability of A or B happening is the probability of A plus the probability of B minus the probability of A and B." This follows from the fact that if A and B occur then that instance is in effect double counted, and so we subtract all of these double counted instances. However, if A and B can not both happen then we say they are disjoint and arrive at

$$ P(A \cup B) = P(A) + P(B).$$

In practice, our $A$ will be $m$ agents in the final hand and $k_0$ agents in the initial draw while $B$ will be $m$ agents in the final hand and $k_1$ agents in the initial draw for different values of $k_0$ and $k_1$. Obviously, since $k_0\neq k_1$, these events are disjoint and $P(A \cap B)=0.$ 
Using these disjoint cases, we arrive at 

$$ P(\text{m Agents}) = \Sigma_{k=0}^{5} P(\text{m Agents}\cap \text{k Agents}).$$

But we have computed these values above via

$$P(\text{m Agents} \cap\text{k Agents}) = P(\text{m Agents}|\text{k Agents})\cdot P(\text{k Agents}),$$

which is essentially the definition.
We thus get the following table of values. Some of the percentages are miniscule. To see more than the first decimal place, change `1` below to the number of decimal places you would like to see.

In [184]:
percentage_decimals = 2

marg_data = []

for m in range(hand_size + 1):
    row = {'m': m}
    
    # Add P(m|k)*P(k) for each k (0 to 5) as numeric values first
    total_prob = 0
    for k in range(hand_size + 1):
        p_m_given_k = calculation_details[m][k]['p_conditional']
        p_k = calculation_details[m][k]['p_k']
        row[f'P(m & {k})'] = p_m_given_k * p_k
        total_prob += p_m_given_k * p_k
    
    # Final column: marginal probability P(m)
    row['P(m)'] = total_prob
    
    marg_data.append(row)

df_marginal = pd.DataFrame(marg_data)

# Format the probability columns as percentages with 1 decimal place
prob_columns = [col for col in df_marginal.columns if col != 'm']
df_marginal_formatted = df_marginal.copy()
for col in prob_columns:
    df_marginal_formatted[col] = df_marginal_formatted[col].apply(format_percent)

print("Marginal Probabilities: P(m agents in final hand)")
print("=" * 140)
print("Each row is a value of m, columns show P(m and k) for k=0 to 5, final column is P(m)")
print("=" * 140)
print(df_marginal_formatted.to_string(index=False))
print("=" * 140)

df_marginal_friendly = df_marginal.copy()


Marginal Probabilities: P(m agents in final hand)
Each row is a value of m, columns show P(m and k) for k=0 to 5, final column is P(m)
 m P(m & 0) P(m & 1) P(m & 2) P(m & 3) P(m & 4) P(m & 5)   P(m)
 0   37.80%   36.58%   12.19%    1.68%    0.09%    0.00% 88.35%
 1    0.00%    7.03%    3.75%    0.58%    0.03%    0.00% 11.40%
 2    0.00%    0.00%    0.21%    0.04%    0.00%    0.00%  0.25%
 3    0.00%    0.00%    0.00%    0.00%    0.00%    0.00%  0.00%
 4    0.00%    0.00%    0.00%    0.00%    0.00%    0.00%  0.00%
 5    0.00%    0.00%    0.00%    0.00%    0.00%    0.00%  0.00%


#### $E[\text{? Agents}|\text{k Agents}]$ and $E[\text{? Agents}]$
We now turn to the computation of expected value. We compute both these conditional expected values and the overall expected value of number of agents. By definition, we have that 

$$E[X|Y=y]=\Sigma_x x\cdot P(X=x|Y=y)$$

where $x$ is the possible values of the random variable $X$. In our case of interest, we would get

$$E[\text{final agent count}|\text{k Agents}]=\Sigma_{m=0}^{m=5} m\cdot P(\text{m Agents}|\text{k Agents}).$$

Change the number of decimals displayed by changing the variable `decimal` in the below cell and executing it.

In [185]:
decimals = 3

In [186]:
# Create a dataframe for conditional expected values
# E[final agents | k agents] = Σ(m=0 to 5) m * P(m|k)

# First, build a dataframe with one row per k, showing P(m|k) for each m
ce_data = []

for k in range(6):
    row = {'k': k}
    
    # Add P(m|k) for each m
    for m in range(6):
        p_m_given_k = calculation_details[m][k]['p_conditional']
        row[f'P({m}|k)'] = p_m_given_k
    
    ce_data.append(row)

df_conditional = pd.DataFrame(ce_data)

# Now add columns for m * P(m|k) and compute conditional expected value
for m in range(6):
    col_name = f'P({m}|k)'
    df_conditional[f'{m}*P({m}|k)'] = m * df_conditional[col_name]

# Sum across for each k to get E[final agents | k]
df_conditional['E[final|k]'] = df_conditional[[f'{m}*P({m}|k)' for m in range(6)]].sum(axis=1)

df_conditional_display = df_conditional.copy()
df_conditional_display=df_conditional_display[['k'] + [f'{m}*P({m}|k)' for m in range(6)] + ['E[final|k]']]

disp_columns = [col for col in df_conditional_display.columns if col != 'k']
for col in disp_columns:
    df_conditional_display[col] = df_conditional_display[col].apply(format_decimals)



print("Conditional Expected Values: E[final agents | k initial agents]")
print("=" * 100)
print(df_conditional_display.to_string(index=False))
print("=" * 100)

df_conditional_friendly = df_conditional.copy()


Conditional Expected Values: E[final agents | k initial agents]
 k 0*P(0|k) 1*P(1|k) 2*P(2|k) 3*P(3|k) 4*P(4|k) 5*P(5|k) E[final|k]
 0    0.000    0.000    0.000    0.000    0.000    0.000      0.000
 1    0.000    0.161    0.000    0.000    0.000    0.000      0.161
 2    0.000    0.232    0.026    0.000    0.000    0.000      0.258
 3    0.000    0.252    0.037    0.001    0.000    0.000      0.290
 4    0.000    0.232    0.026    0.000    0.000    0.000      0.258
 5    0.000    0.161    0.000    0.000    0.000    0.000      0.161


In [187]:

print(f"E[final agents] = {df_conditional['E[final|k]'].sum():.3f} agents")


E[final agents] = 1.129 agents


## Draw 7 Keep 5

This mulligan system is simpler than the Friendliest Mulligan: the player draws 7 cards and then must discard exactly 2 cards. There is no redrawing—the discarded cards are simply shuffled back into the deck. Again, assuming the strategy of minimizing the number of agents, the player will discard agents first, and only discard non-agents if there are not enough agents to discard.

After drawing 7 cards with $k$ agents, the player discards exactly 2 cards. In order to minimize agents, discarding agents is prioritized. This means:

- If $k \geq 2$: You discard 2 agents, leaving $k - 2$ agents in your final hand
- If $k = 1$: You discard the 1 agent and 1 non-agent, leaving 0 agents
- If $k = 0$: You discard 2 non-agents, leaving 0 agents

In general, the number of agents in your final hand is $m = \max(0, k - 2)$, which is a deterministic function of $k$. Therefore, $P(\text{m Agents}|\text{k Agents})$ is either 0 or 1 and so we will not need to do extensive computations.

Change the number of decimals that are displayed for percentages by changing the value of the variable `percentage_decimals` below and executing it.

In [188]:
# Set up percentage formatting function
percentage_decimals = 4

In [189]:
def compute_initial_draw_probabilities(total, marked, initial_draw):
    k_values = range(0, min(marked, initial_draw) + 1)
    probabilities = {}
    
    for k in k_values:
        num = comb(marked, k, exact=True) * comb(total - marked, initial_draw - k, exact=True)
        denom = comb(total, initial_draw, exact=True)
        prob = num / denom
        probabilities[k] = prob
    
    return probabilities


def build_initial_draw_dataframe(k_values, probabilities):
    return pd.DataFrame({
        'k (Agents)': k_values,
        'Probability': [format_percent(p) for p in probabilities.values()]
    })


def compute_expected_value_initial_draw(k_values, probabilities):
    return sum(probabilities[k] * k for k in k_values)


def run_initial_draw_analysis(total, marked, initial_draw):
    """
    Run complete initial draw analysis: compute probabilities, display table, and expected value.
    """
    # Compute probabilities
    k_values = range(0, min(marked, initial_draw) + 1)
    probabilities = compute_initial_draw_probabilities(total, marked, initial_draw)
    
    # Create and display DataFrame
    df_initial = build_initial_draw_dataframe(k_values, probabilities)
    print("Initial Draw Probabilities (Draw 7)")
    print("=" * 80)
    print(df_initial.to_string(index=False))
    print("=" * 80)
    
    # Compute and display expected value
    expected_value = compute_expected_value_initial_draw(k_values, probabilities)
    print(f"\nExpected value in initial draw = {expected_value:.3f} agents")
    print("=" * 80)
    
    return probabilities, expected_value


# Run analysis for Draw 7
probabilities_7, expected_value_7 = run_initial_draw_analysis(
    total=36,
    marked=6,
    initial_draw=7
)

Initial Draw Probabilities (Draw 7)
 k (Agents) Probability
          0    24.3876%
          1    42.6783%
          2    25.6070%
          3     6.5659%
          4     0.7295%
          5     0.0313%
          6     0.0004%

Expected value in initial draw = 1.167 agents


#### $P(\text{m Agents})$ - Draw 7 Keep 5

The process of discarding $2$ cards and not redrawing means that some of the probabilities aggregate as follows.

- $P(\text{0 Agents in final hand})=P(\text{initial Agents} \leq2).$ Thus we have that $P(m=0)=P(k=0)+P(k=1)+P(k=2).$
- For $0<j<5$ we have that $P(m=j)=P(k=j+2).$
- The max number of possible agents in a final hand is $4$ and this can only happen if you draw all $6$ agents in the intial $7$ cards.


In [190]:
def compute_k_to_m_transformation(probabilities, discard_count):
    """Apply deterministic transformation m = max(0, k - discard_count)."""
    transform_data = []
    prob_by_final = {}
    
    for k in probabilities.keys():
        m = max(0, k - discard_count)
        p_k = probabilities[k]
        
        transform_data.append({
            'k (Drawn)': k,
            'm (Final)': m,
            'P(k)': p_k,
            'P(k) %': format_percent(p_k)
        })
        
        if m not in prob_by_final:
            prob_by_final[m] = 0.0
        prob_by_final[m] += p_k
    
    return transform_data, prob_by_final


def build_transformation_dataframe(transform_data):
    """Build DataFrame showing k -> m transformation."""
    return pd.DataFrame(transform_data)


def build_final_distribution_dataframe(prob_by_final):
    """Build DataFrame of final probability distribution."""
    final_dist_data = []
    for m in sorted(prob_by_final.keys()):
        final_dist_data.append({
            'm (Final)': m,
            'P(m)': prob_by_final[m],
            'P(m) %': format_percent(prob_by_final[m])
        })
    return pd.DataFrame(final_dist_data)


def run_final_agent_distribution_analysis(probabilities, discard_count=2):
    """Compute and display final agent distribution after discarding cards."""
    print("\nFinal Agent Probabilities - Draw 7 Keep 5")
    print("=" * 100)
    print(f"After drawing {sum(1 for _ in probabilities)} cards and discarding {discard_count}, "
          f"the transformation is: m = max(0, k - {discard_count})")
    print("=" * 100)
    
    # Compute transformation
    transform_data, prob_by_final = compute_k_to_m_transformation(probabilities, discard_count)
    
    # Display transformation table
    df_transform = build_transformation_dataframe(transform_data)
    print("\nTransformation: Initial Agents → Final Agents")
    print("-" * 100)
    print(df_transform[['k (Drawn)', 'm (Final)', 'P(k) %']].to_string(index=False))
    
    # Display final distribution
    print("\n" + "=" * 100)
    print("Final Distribution: Probability of m agents in final hand")
    print("=" * 100)
    
    df_final = build_final_distribution_dataframe(prob_by_final)
    print(df_final[['m (Final)', 'P(m) %']].to_string(index=False))
    print("=" * 100)
    df_final['m'] = df_final['m (Final)']
    return prob_by_final, df_transform, df_final


# Run analysis for Draw 7/Keep 5
prob_by_final, df_transform, df_final7= run_final_agent_distribution_analysis(
    probabilities=probabilities_7,
    discard_count=2
)




Final Agent Probabilities - Draw 7 Keep 5
After drawing 7 cards and discarding 2, the transformation is: m = max(0, k - 2)

Transformation: Initial Agents → Final Agents
----------------------------------------------------------------------------------------------------
 k (Drawn)  m (Final)   P(k) %
         0          0 24.3876%
         1          0 42.6783%
         2          0 25.6070%
         3          1  6.5659%
         4          2  0.7295%
         5          3  0.0313%
         6          4  0.0004%

Final Distribution: Probability of m agents in final hand
 m (Final)   P(m) %
         0 92.6729%
         1  6.5659%
         2  0.7295%
         3  0.0313%
         4  0.0004%


#### $E[\text{? Agents}]$ - Draw 7 Keep 5

This can be computed directly following the definition. 

In [191]:

# Calculate expected value
expected_final = sum(m * prob_by_final[m] for m in prob_by_final.keys())
print(f"\nExpected number of agents in final hand: {expected_final:.3f}")


Expected number of agents in final hand: 0.081


<!-- DOCUMENTATION CONSISTENCY CHECK: The notebook contains two different docstring styles. Early sections have verbose docstrings with Parameters/Returns sections (e.g., compute_pk_unique, compute_e_desired, compute_p_m_given_k). Later refactored sections use brief one-line docstrings (e.g., compute_ev_comparison: "Extract expected values from both strategies."). Consider standardizing: either use concise docstrings throughout or add detailed parameter documentation to the refactored functions. -->

<!-- ORGANIZATION CHECK: This section begins with just a one-line introduction "We are now in a position to compare the two different methods." followed immediately by code cells. Unlike other major sections which have explanatory context/setup before code, this section lacks any explanation of what the comparison will show or what metrics will be used. Consider adding a brief explanation like: "We will now compare the Friendliest Mulligan and Draw 7 Keep 5 strategies, examining both the probability distributions and expected values of agents in the final hand." -->

## Comparison

We are now in a position to compare the two different methods. We will compare both $P(\text{Agents=m})$ and $P(\text{Agents}<= m)$.

In [192]:
def build_distribution_comparison_dataframe(df_friendly, df_draw7,  label, num_values=6):
    """Build DataFrame comparing distributions from two mulligan strategies."""
    comparison_data = []
    
    for m in range(num_values):
        p_m_friendly = df_friendly[df_friendly['m'] == m]['P(m)'].values[0] \
                       if len(df_friendly[df_friendly['m'] == m]) > 0 else 0.0
        p_m_draw7 = df_draw7[df_draw7['m'] == m]['P(m)'].values[0] \
                    if len(df_draw7[df_draw7['m'] == m]) > 0 else 0.0
        diff = p_m_draw7 - p_m_friendly
        
        comparison_data.append({
            label: m,
            'Friendliest %': format_percent(p_m_friendly),
            'Draw 7 Keep 5 %': format_percent(p_m_draw7),
            'Difference %': format_percent(diff) if abs(diff) > 1e-10 else "—"
        })
    
    return pd.DataFrame(comparison_data)


def compute_ev_comparison(df_conditional, expected_final):
    """Extract expected values from both strategies."""
    ev_friendly = df_conditional['E[final|k]'].sum()
    ev_draw7 = expected_final
    return ev_friendly, ev_draw7


def determine_better_strategy(ev_friendly, ev_draw7):
    """Determine which strategy reduces agents more and return message."""
    if ev_friendly < ev_draw7:
        return f"Friendliest Mulligan is BETTER (reduces agents by {ev_friendly - ev_draw7:.3f})"
    elif ev_draw7 < ev_friendly:
        return f"Draw 7 Keep 5 is BETTER (reduces agents by {ev_draw7 - ev_friendly:.3f})"
    else:
        return "Both strategies have EQUAL effectiveness"

labels = ["COMPARISON: Final Agent Distribution P(m agents)", "COMPARISON: Expected Number of Agents in Final Hand", "m (Agents)"]
def run_mulligan_strategy_comparison(df_marginal, df_draw7, df_conditional, expected_final, labels):
    """Compare two mulligan strategies on distribution and expected value."""
    # Distribution comparison
    print("\n" + "=" * 120)
    print(labels[0])
    print("=" * 120)
    
    df_comparison = build_distribution_comparison_dataframe(df_marginal, df_draw7, labels[2], 6)
    print(df_comparison.to_string(index=False))
    print("=" * 120)
    
    # Expected value comparison
    print("\n" + "=" * 120)
    print(labels[1])
    print("=" * 120)
    
    ev_friendly, ev_draw7 = compute_ev_comparison(df_conditional, expected_final)
    
    print(f"{'Strategy':<25} {'Expected Value':>20} {'Comparison':>30}")
    print("-" * 120)
    print(f"{'Friendliest Mulligan':<25} {ev_friendly:>20.3f}")
    print(f"{'Draw 7 Keep 5':<25} {ev_draw7:>20.3f}")
    print("-" * 120)
    print(f"{'Difference (Draw 7 - Friendly)':<25} {ev_draw7 - ev_friendly:>+20.3f}")
    print("=" * 120)
    
    # Determine better strategy
    result = determine_better_strategy(ev_friendly, ev_draw7)
    print(f"\n{result}")
    


# Run comparison
run_mulligan_strategy_comparison(
    df_marginal_friendly, df_final7, df_conditional_friendly, expected_final, labels
)


COMPARISON: Final Agent Distribution P(m agents)
 m (Agents) Friendliest % Draw 7 Keep 5 % Difference %
          0      88.3494%        92.6729%      4.3236%
          1      11.3970%         6.5659%     -4.8311%
          2       0.2531%         0.7295%      0.4764%
          3       0.0005%         0.0313%      0.0308%
          4       0.0000%         0.0004%      0.0004%
          5       0.0000%         0.0000%            —

COMPARISON: Expected Number of Agents in Final Hand
Strategy                        Expected Value                     Comparison
------------------------------------------------------------------------------------------------------------------------
Friendliest Mulligan                     1.129
Draw 7 Keep 5                            0.081
------------------------------------------------------------------------------------------------------------------------
Difference (Draw 7 - Friendly)               -1.048

Draw 7 Keep 5 is BETTER (reduces agents by -1

For the cumulative distributions, we use that 
$$P(X\leq a) = \Sigma_{n=0}^a P(X=n)$$
for a discrete distribution.

In [193]:
# Simple function for computing CDF from a dataframe with P(m) values
def compute_cdf(df, value_col='m (Agents)', prob_col='P(m)'):
    """
    Compute cumulative distribution P(Agents ≤ m) from a PMF dataframe.
    
    Assumes df is sorted by value_col (or will sort it internally).
    Returns a copy with new column 'P(m ≤ Agents)' containing cumulative probabilities.
    """
    df_cdf = df.copy()
    df_cdf = df_cdf.sort_values(by=value_col).reset_index(drop=True)
    df_cdf['P(m ≤ Agents)'] = df_cdf[prob_col].cumsum()
    return df_cdf


# EXAMPLE: If you have df_final with P(m) probabilities
if 'df_final' in dir() and 'P(m)' in df_final.columns:
    df_cdf_result = compute_cdf(df_final, value_col='m (Final)', prob_col='P(m)')
    print("\nCumulative Distribution: P(Agents ≤ m)")
    print("=" * 100)
    print(df_cdf_result.to_string(index=False))
    print("=" * 100)


Cumulative Distribution: P(Agents ≤ m)
 m (Final)     P(m)   P(m) %  m  P(m ≤ Agents)
         0 0.926729 92.6729%  0       0.926729
         1 0.065659  6.5659%  1       0.992388
         2 0.007295  0.7295%  2       0.999684
         3 0.000313  0.0313%  3       0.999996
         4 0.000004  0.0004%  4       1.000000


In [194]:
# Apply CDF computation to both strategies in the comparison
# Extract numerical probabilities from the comparison dataframe
df_comparison_copy = df_comparison.copy()

# Convert percentage strings to floats for Friendliest Mulligan
df_comparison_copy['Friendliest'] = df_comparison_copy['Friendliest %'].str.rstrip('%').astype(float) / 100

# Convert percentage strings to floats for Draw 7 Keep 5
df_comparison_copy['Draw 7'] = df_comparison_copy['Draw 7 Keep 5 %'].str.rstrip('%').astype(float) / 100

# Compute CDFs
df_comparison_copy['CDF Friendly'] = df_comparison_copy['Friendliest'].cumsum()
df_comparison_copy['CDF Draw 7'] = df_comparison_copy['Draw 7'].cumsum()

# Display comparison with CDFs
print("\nComparison with Cumulative Distributions")
print("=" * 140)
print(df_comparison_copy[['m (Agents)', 'Friendliest %', 'CDF Friendly', 
                           'Draw 7 Keep 5 %', 'CDF Draw 7']].to_string(index=False))
print("=" * 140)
print("\nInterpretation:")
print("  CDF Friendly: P(Agents ≤ m) for Friendliest Mulligan")
print("  CDF Draw 7: P(Agents ≤ m) for Draw 7 Keep 5 strategy")
print("=" * 140)
# It would be nice if the columns were spaced better.


Comparison with Cumulative Distributions
 m (Agents) Friendliest %  CDF Friendly Draw 7 Keep 5 %  CDF Draw 7
          0      88.3494%      0.883494        92.6729%    0.926729
          1      11.3970%      0.997464         6.5659%    0.992388
          2       0.2531%      0.999995         0.7295%    0.999683
          3       0.0005%      1.000000         0.0313%    0.999996
          4       0.0000%      1.000000         0.0004%    1.000000
          5       0.0000%      1.000000         0.0000%    1.000000

Interpretation:
  CDF Friendly: P(Agents ≤ m) for Friendliest Mulligan
  CDF Draw 7: P(Agents ≤ m) for Draw 7 Keep 5 strategy


# Hard Mulligan for specific cards
The previous sections focused on minimizing the number of agents in your starting hand. However, there are other goals that a player may be working towards. One such goal might be to get specific set up cards in your opening hand. Our strategy will be to mulligan as much as possible in order to get the desired cards in case they are not in our initial draw.

## Problem Setup

We are interested in maximizing the number of **desired cards** in our opening hand. The approach is:

**Desired Cards Definition:**
- Let $S$ be a set of $unique_desired$ specific cards we want in our hand.
- Let $desired_count$ be the number of copies of each of these desired cards.
- The total number of desired cards is then $unique_desired\cdot desired_count$.

**Note:** We could instead work with a mapping $f$ where $f(i)$ is the number of copies of card $i$ in the deck. The computation in this case may be approached later due to complexity issues.

**Discard Strategy:**
Given an initial draw of 5 cards:
1. For each desired card $i$ where we have $m$ copies, **discard $m-1$ of them** (keep 1)
2. For undesired cards, **discard them if we don't already have one copy of each desired card**

**Quantities of Interest:**
- $P(\text{j desired cards})$ for various $j$.
- - This will allow us to compute $P(\text{at least } j \text{ desired cards}).$
- $E[\text{number of desired cards}]$ in the final 5-card hand after mulligan

**Note**: The case where we are not concerned with uniqueness of the desired cards is a special case of this situation. We get the result by setting $desired_count=1$ and considering each card as unique. We can see that this computes the desired value by considering how our mulligan strategy is effected: we don't discard any card from this pool of desired cards.

## 'Friendliest Mulligan'
### Initial Draw Computations

We first must compute the probability of the initial draw having $k$ distinct desired cards. We will denote this as $P(\text{k Unique}).$


- **$k$**: number of distinct desired cards drawn ($0$ to $max(uniquq_desired, hand_size)$).

#### Computation outline
We will make use of the inclusion exclusion principle to compute $P(\text{k Unique})$, the probability of having exactly k unique desired cards in an opening hand. This does not mean we do not have extra copies of one of these $k$ cards, just that the extra copy of a desired card will not contribute to $k$. We will refer to the different unique cards as types. So when we say we have $2$ cards of the same type this will mean that we have two copies of the same card. 

The core of the inclusion-exclusion principle is that you have an initial count which is over, and then you correct by removing cases. However, you removed too many so you need to add some cases back. This process is repeated and when organized properly is controllable and will terminate.

The final formula will be 

$$P(\text{k Unique})=\frac{\binom{3}{k}}{\binom{36}{5}}\Sigma_{j=0}^k(-1)^j\binom{k}{j}\binom{2(k-j)+30}{5}$$

- $\binom{3}{k}$ counts the number of different ways we can choose the types of the cards.
- $\binom{k}{j}$ counts the number of ways we can choose $j$ of the $k$ types for our hand to not include.
- $\binom{2(k-j)+30}{5}$ counts the number of ways this hand can be drawn.
- $j=0$ counts all ways to draw from the $k$ types + undesired cards. This counts hands that include no desired cards, for example.
- $j=1$ counts all ways to draw an initial hand that miss at least one type of card. These hands are removed $\binom{k}{1}=k$ times.
- $j=2$ counts all ways to draw an initial hand that miss at least two types of cards. These are added back because they were removed too many times in the previous step.
- In general, for odd $j$ we remove hands and for even $j$ we add hands back.
- We continue in this way until $j=k$ which will address hands that have no desired cards. Whether or not these should be removed is  dependent on whether or not $k$ is even or odd.
- In the end, we will be left with hands that will contain exactly $k$ unique types.

##### Example for $k=2$
Lets work through the above formula in the specific example of $k=2$. The above formula becomes

$$P(\text{\underline{2} Unique})=\frac{\binom{3}{\underline{2}}}{\binom{36}{5}}\Sigma_{j=0}^{\underline{2}}(-1)^j\binom{\underline{2}}{j}\binom{2(\underline{2}-j)+30}{5}.$$

We have underlined $2$ to keep track of when the value is $2$ because $k=2$.
We will look at each term in the sum and consider what is being counted as $j$ ranges from $0$ to $2$. The term $\frac{\binom{3}{\underline{2}}}{\binom{36}{5}}$ addresses the different ways these $\underline{2}$ unique cards could be chosen from among $unique_desired=3$ cards so we will not address this term. So we can assume that we have already selected the $2$ types of unique cards that we are considering. Let's call these types $A$ and $B$. This restricts our larger card pool. We are now only going to consider hands where the cards are either one of the $2\cdot \underline{2}=4$ desired cards we have selected or one of the $total-unique_desired\cdot desired_count=30$ undesired cards. $j$ will be counting how many of our desired types we are excluding from the count.

- $j=0$: we have the contribution 
$$(-1)^0\binom{\underline{2}}{0}\binom{2(\underline{2}-0)+30}{5}=1\cdot1\cdot\binom{34}{5}.$$
The first $1$ means that we are starting with a positive number of hands that we are considering. The second $1$ comes from the fact that there is only $1$ way to not exclude any of the types from the hands we are considering. Since $j=0$, we are going to consider all hands from the restricted pool of $\binom{34}{5}$ cards. This will include hands where type $A$ and $B$ are present as well as hands were neither are present or where only 1 type is present. Thus we see we are over counting and we need to remove some of these hands.
- $j=1$: we have the contribution 
$$(-1)^1\binom{\underline{2}}{1}\binom{2(\underline{2}-1)+30}{5}=-1\cdot2\cdot\binom{32}{5}.$$
In the previous term, we counted too many hands. We counted hands that didn't have $A$ or $B$ or either. As $j=1,$ we will now remove the hands that are missing either type $A$ or type $B$. Since we are removing hands from the count we have the initial $-1$. Since we could remove hands that exclude $A$ or hands that exclude $B$, we must multiply the numebr of hands we remove by $\binom{\underline{2}}{1}=2$. Finally, we are considering hands that are made from the restricted set of $\binom{2(\underline{2}-1)+30}{5}=\binom{32}{5}$ cards. Note that some of the possible hands will not contain a card of the desired type. These hands occur twice in this collection that we remove because we multiply the count by $2$. So we have removed these hands too many times and will need to add them back.

- $j=2$: we have the contribution 
$$(-1)^2\binom{\underline{2}}{2}\binom{2(\underline{2}-2)+30}{5}=1\cdot 1\cdot\binom{30}{5}.$$
In the previous term, we removed too many hands. We counted hands that did not have a card of type $A$ and then we multiplied by $2$ in order to also count hands that did not contain a card of type $B$. However, some of these hands that didn't contain a card of type $A$ also didn't contain a card of type $B$, so we effectively removed those hands twice instead of just once. This is because hands that don't have any cards of type $A$ or of type $B$ both count as not having a card of type $A$ as well as not having a card of type $B$. In order to correct this, we must add these hands back. When $j=2$, we are considering the hands that have no cards of type $A$ as well as no cards of type $B$. We are adding cards back and this explains the first $1$. The second $1$ is there because there is only a single way to exclude $2$ cards from the count. Our restricted pool now has only $2(\underline{2}-2)+30=30$ cards in it. The number of hands that can be made from this are $\binom{30}{5}.$

A similar analysis can be done for other values of $k$ to better understand the inclusion-exclusion principle.

#### $P(\text{ k Unique})$ Computation
We now return to the computation at hand. Below, we compute the probability of having $k$ unique desired cards in your initial draw. This includes an initial draw where you have multiple copies of one of the desired cards. It is simply that the other copies to not contribute to the count $k$. This computation can be adjusted by changing the value of $unique_desired$ which is the number of unique desired cards and $desired_count$ which is the number of copies of each desired card. Note that changing these values will impact the computation

**Note**: A more interesting and complicated question is when the $desired_count$ variable is actually a non-constant value. Perhaps this computation will be implemented later.

In [195]:



# Direct Computation of P(k unique) using Inclusion-Exclusion Principle

from scipy.special import comb
import pandas as pd
hand_size=5
unique_desired = 3 # Number of unique desired cards
desired_count = 2
total_cards = 36
num_desired = desired_count * unique_desired
num_undesired = total_cards - num_desired

total_ways = comb(total_cards, hand_size, exact=True)

def build_inclusion_exclusion_table(unique_desired, desired_count, num_undesired, total_cards, hand_size):
    # Build dataframe with one row per k value
    data = []

    for k in range(unique_desired + 1):
        row = {'k': k}
    
        # Compute each j-term in the inclusion-exclusion sum
        total_sum = 0
        total_ways = comb(total_cards, hand_size, exact=True)
        for j in range(unique_desired + 1):  # Make square matrix by including all j values even if 0
            if j <= k:
                # Compute: (-1)^j * C(k,j) * C(2(k-j) + undesired, hand_size)
                sign = (-1) ** j
                comb_k_j = comb(k, j, exact=True)
                available_desired = desired_count * (k - j)
                available_total = available_desired + num_undesired
                comb_available = comb(available_total, hand_size, exact=True)
            
                term = sign * comb_k_j * comb_available
                total_sum += term
            else:
                term = 0
        
            row[f'j={j}'] = term
    
        # Compute P(k) = C(3,k) / C(36,5) * total_sum
        ways_choose_k_types = comb(unique_desired, k, exact=True)
        
        prob_k = (ways_choose_k_types * total_sum) / total_ways
        row['P(k)'] = prob_k
        row['P(k) %'] = f"{prob_k * 100:.4f}%"
    
        data.append(row)

    return pd.DataFrame(data)

def compute_pk_desired(k, unique_desired, desired_count, num_undesired, total_cards, hand_size):
    num_desired = unique_desired * desired_count
    comb_desired = comb(num_desired, k, exact=True)
    comb_undesired = comb(num_undesired, hand_size - k, exact=True)
    total_ways = comb(total_cards, hand_size, exact=True)
    if total_ways == 0:
        return 0.0
    prob_k = (comb_desired * comb_undesired) / total_ways
    return prob_k

def compute_pk_unique(k, unique_desired, desired_count, num_undesired, total_cards, hand_size):
    total_sum = 0
    for j in range(unique_desired + 1):
        if j <= k:
            sign = (-1) ** j
            comb_k_j = comb(k, j, exact=True)
            available_desired = desired_count * (k - j)
            available_total = available_desired + num_undesired
            comb_available = comb(available_total, hand_size, exact=True)
        
            term = sign * comb_k_j * comb_available
            total_sum += term
    
    ways_choose_k_types = comb(unique_desired, k, exact=True)
    total_ways = comb(total_cards, hand_size, exact=True)
    
    # Handle impossible scenario (e.g., hand_size > total_cards)
    if total_ways == 0:
        return 0.0
    
    prob_k = (ways_choose_k_types * total_sum) / total_ways
    return prob_k


def present_inclusion_exclusion_results(df):
    print("Direct Computation of P(k unique) using Inclusion-Exclusion Principle")
    print("=" * 140)
    print("Each row: k unique desired cards in initial draw")
    print("Columns j=0 to j=3: contribution from (-1)^j * C(k,j) * C(2(k-j)+30, 5)")
    print("Column P(k): final probability = C(3,k)/C(36,5) * (sum of j terms)")
    print("=" * 140)
    print(df.to_string(index=False))
    print("=" * 140)
    print(f"\nVerification: Sum of P(k) = {df['P(k)'].sum():.10f} (should be 1.0)")
    print()

df_pk_direct = build_inclusion_exclusion_table(unique_desired, desired_count, num_undesired, total_cards, hand_size)
present_inclusion_exclusion_results(df_pk_direct)

Direct Computation of P(k unique) using Inclusion-Exclusion Principle
Each row: k unique desired cards in initial draw
Columns j=0 to j=3: contribution from (-1)^j * C(k,j) * C(2(k-j)+30, 5)
Column P(k): final probability = C(3,k)/C(36,5) * (sum of j terms)
 k    j=0     j=1    j=2     j=3     P(k)   P(k) %
 0 142506       0      0       0 0.378008 37.8008%
 1 201376 -142506      0       0 0.468471 46.8471%
 2 278256 -402752 142506       0 0.143319 14.3319%
 3 376992 -834768 604128 -142506 0.010202  1.0202%

Verification: Sum of P(k) = 1.0000000000 (should be 1.0)



<!-- VOICE CHECK: Shifts from "We repeat" (collective documentation) to "we simulate" (still collective) — unclear if "we" includes reader or is pure authorial/mathematical "we" -->
We repeat the above with the different values for `unique_desired` and `desired_count`. With the following values we simulate the number of agents in an opening hand.

In [196]:
unique_desired = 6
desired_count = 1
num_desired = desired_count * unique_desired
num_undesired = total_cards - num_desired
df_pk_direct_6 = build_inclusion_exclusion_table(unique_desired, desired_count, num_undesired, total_cards, hand_size)
present_inclusion_exclusion_results(df_pk_direct_6)

Direct Computation of P(k unique) using Inclusion-Exclusion Principle
Each row: k unique desired cards in initial draw
Columns j=0 to j=3: contribution from (-1)^j * C(k,j) * C(2(k-j)+30, 5)
Column P(k): final probability = C(3,k)/C(36,5) * (sum of j terms)
 k    j=0      j=1     j=2      j=3     j=4      j=5    j=6     P(k)   P(k) %
 0 142506        0       0        0       0        0      0 0.378008 37.8008%
 1 169911  -142506       0        0       0        0      0 0.436163 43.6163%
 2 201376  -339822  142506        0       0        0      0 0.161542 16.1542%
 3 237336  -604128  509733  -142506       0        0      0 0.023077  2.3077%
 4 278256  -949344 1208256  -679644  142506        0      0 0.001194  0.1194%
 5 324632 -1391280 2373360 -2013760  849555  -142506      0 0.000016  0.0016%
 6 376992 -1947792 4173840 -4746720 3020640 -1019466 142506 0.000000  0.0000%

Verification: Sum of P(k) = 1.0000000000 (should be 1.0)




And now we increase the number of unique desired cards to $4$ with $3$ copies of each.

In [197]:
unique_desired = 4
desired_count = 3
num_desired = desired_count * unique_desired
num_undesired = total_cards - num_desired
df_pk_direct_6 = build_inclusion_exclusion_table(unique_desired, desired_count, num_undesired, total_cards, hand_size)
present_inclusion_exclusion_results(df_pk_direct_6)

Direct Computation of P(k unique) using Inclusion-Exclusion Principle
Each row: k unique desired cards in initial draw
Columns j=0 to j=3: contribution from (-1)^j * C(k,j) * C(2(k-j)+30, 5)
Column P(k): final probability = C(3,k)/C(36,5) * (sum of j terms)
 k    j=0     j=1    j=2     j=3   j=4     P(k)   P(k) %
 0  42504       0      0       0     0 0.112745 11.2745%
 1  80730  -42504      0       0     0 0.405590 40.5590%
 2 142506 -161460  42504       0     0 0.374809 37.4809%
 3 237336 -427518 242190  -42504     0 0.100840 10.0840%
 4 376992 -949344 855036 -322920 42504 0.006016  0.6016%

Verification: Sum of P(k) = 1.0000000000 (should be 1.0)



#### Expected Value of initial draw

We now compute both the expected number of desired cards and the expected number of unique desired cards in an initial draw. We first build the input for our expected value computation. We will then compute the expected number of desired cards as well as the expected number of unique desired cards.

In [198]:
# Define scenarios
scenarios = [
        {'unique_desired': 1, 'desired_count': 2},
        {'unique_desired': 2, 'desired_count': 2},
        {'unique_desired': 3, 'desired_count': 2},
        {'unique_desired': 4, 'desired_count': 2},
        {'unique_desired': 5, 'desired_count': 2},
        {'unique_desired': 4, 'desired_count': 3},
        {'unique_desired': 6, 'desired_count': 1},
    ]

def max_scenario_parameters(scenarios):
    max_unique_desired = max(scenario['unique_desired'] for scenario in scenarios)
    max_desired_count = max(scenario['desired_count'] for scenario in scenarios)
    return max_unique_desired, max_desired_count

def build_pk_unique_dataframe_no_expectations(scenarios, total_cards, hand_size):  
    data = []
    for scenario in scenarios:
        unique_desired = scenario['unique_desired']
        desired_count = scenario['desired_count']
        
        # Compute number of undesired cards
        n_undesired = total_cards - (unique_desired * desired_count)
        
        # Build row for this scenario
        row = {
            'unique_desired': unique_desired,
            'desired_count': desired_count,
        }
        
        # Add P(k) for each k from 0 to unique_desired using compute_pk_unique
        for k in range(unique_desired + 1):
            p_k = compute_pk_unique(k, unique_desired, desired_count, n_undesired, total_cards, hand_size)
            row[f'P(k={k})'] = p_k
        
        # Add 0's for k values beyond unique_desired (to ensure consistent columns across scenarios)
        max_unique_desired, _ = max_scenario_parameters(scenarios)
        for k in range(unique_desired + 1, max_unique_desired + 1):  # Up to k=6 to match max scenario
            row[f'P(k={k})'] = 0.0
        data.append(row)
    return pd.DataFrame(data)

def compute_e_unique_for_row(row):
    """
    Compute E[unique desired cards] for a single row.
    
    Parameters:
    -----------
    row : dict or pandas.Series
        A row from build_pk_dataframe_no_expectations() output.
        Must contain 'unique_desired' and 'P(k=...)' columns.
    
    Returns:
    --------
    float : E[unique desired cards] = sum of k * P(k unique = k)
    """
    
    unique_desired = int(row['unique_desired'])
    e_unique = 0.0
    
    for k in range(unique_desired + 1):
        p_k_col = f'P(k={k})'
        if p_k_col in row:
            p_k = row[p_k_col]
        else:
            p_k = 0.0
        
        # E[unique | k] = k (by definition if we drew k unique types)
        e_unique += k * p_k
    
    return e_unique

def compute_e_desired(unique_desired, desired_count, total_cards, hand_size):
    
    
    # Convert to int (in case they come from numpy/pandas as float64)
    unique_desired = int(unique_desired)
    desired_count = int(desired_count)
    
    # Total number of desired cards in the deck
    num_desired_total = unique_desired * desired_count
    num_undesired = total_cards - num_desired_total
    
    e_desired = 0.0
    
    # Compute E[desired] over all possible k (number of desired cards drawn)
    for k in range(min(hand_size, num_desired_total) + 1):
        p_k_desired = compute_pk_desired(k, unique_desired, desired_count, num_undesired, total_cards, hand_size)
        e_desired += k * p_k_desired
    
    return e_desired

def adjoin_expected_values(df_pk, hand_size_param=None):
    """
    Adjoin 'E[desired]' and 'E[unique]' columns to a dataframe.
    
    Parameters:
    -----------
    df_pk : DataFrame
        Output from build_pk_dataframe_no_expectations()
    hand_size_param : int, optional
        Hand size to use for computing expectations. If None, uses global hand_size.
    
    Returns:
    --------
    df_with_expectations : DataFrame
        Original dataframe with two additional columns: 'E[desired]' and 'E[unique]'
    """
    
    df_result = df_pk.copy()
    
    # Use provided hand_size or fall back to global
    hs = hand_size_param if hand_size_param is not None else hand_size
    
    # Compute E[unique] for each row
    e_unique_values = []
    for idx, row in df_pk.iterrows():
        e_unique = compute_e_unique_for_row(row)
        e_unique_values.append(e_unique)
    
    # Compute E[desired] for each row
    e_desired_values = []
    for idx, row in df_pk.iterrows():
        e_desired = compute_e_desired(row['unique_desired'], row['desired_count'], total_cards, hs)
        e_desired_values.append(e_desired)
    
    # Add columns to result dataframe
    df_result['E[desired]'] = e_desired_values
    df_result['E[unique]'] = e_unique_values
    
    return df_result


# Build the complete initial draw summary table
df_initial = build_pk_unique_dataframe_no_expectations(scenarios, total_cards, hand_size)

print("P(k unique) Distributions for Each Scenario with expected values")
print("=" * 100)
df_complete = adjoin_expected_values(df_initial)
print(df_complete.to_string(index=False))
print("=" * 100)


P(k unique) Distributions for Each Scenario with expected values
 unique_desired  desired_count   P(k=0)   P(k=1)   P(k=2)   P(k=3)   P(k=4)   P(k=5)  P(k=6)  E[desired]  E[unique]
              1              2 0.738095 0.261905 0.000000 0.000000 0.000000 0.000000     0.0    0.277778   0.261905
              2              2 0.534165 0.407860 0.057975 0.000000 0.000000 0.000000     0.0    0.555556   0.523810
              3              2 0.378008 0.468471 0.143319 0.010202 0.000000 0.000000     0.0    0.833333   0.785714
              4              2 0.260695 0.469251 0.233066 0.035714 0.001273 0.000000     0.0    1.111111   1.047619
              5              2 0.174486 0.431044 0.311041 0.077402 0.005942 0.000085     0.0    1.388889   1.309524
              4              3 0.112745 0.405590 0.374809 0.100840 0.006016 0.000000     0.0    1.666667   1.481793
              6              1 0.378008 0.436163 0.161542 0.023077 0.001194 0.000016     0.0    0.833333   0.833333


<!-- NOTATION CLARIFICATION NEEDED: This section introduces $\text{m Unique}_f$ and $\text{k Unique}_i$ for the first time. The subscripts (f for "final", i for "initial") are helpful but should be explicitly defined before use. Earlier sections used $P(\text{k Unique})$ without subscripts. Consider adding a sentence like: "We adopt the notation $P(\text{k Unique}_i)$ to denote initial draw probabilities and $P(\text{m Unique}_f)$ for final hand probabilities, with subscripts i for 'initial' and f for 'final'." -->

### Discard and Redraw Computations
We now turn to computing $P(\text{m Unique}_f)$. By $\text{m Unique}_f$ we mean that the final hand has $m$ unique desired cards. We will also use the notation $\text{k Unique}_i$ to denote case where the initial draw has $k$ unique desired cards. This means that our previous computations of $P(\text{k Unique})$ will now be denoted by $P(\text{k Unique}_i).$ We will follow the same strategy as before by computing conditional probabilities based on the initial draw. This gives us the formula

$$P(\text{m Unique}_f) = \Sigma_k P(\text{m Unique}_f|\text{k Unique}_i)\cdot P(\text{k Unique}_i).$$

#### Computing $P(\text{m Unique}_f|\text{k Unique}_i)$
At the core of this computation is a Markov chain, but we hide that and explain the scenario as follows.
Here is our process.

- Draw our initial hand of cards.
- Decide which unique desired cards to keep, obviously $k$.
- Redraw $5-k$ cards and count how many of them are unique.

When we computed $P(\text{k Unique}_i)$ above, we did this by drawing cards from a desirable set and an undesirable set. We will compute the number of new desired cards by enlarging the set of undesired cards to now include the desired cards we kept. Again, we will rely on the inclusion exclusion principle. We will use an intermediate formula:

$$Ways(x) =\Sigma_{i=0}^x(-1)^i\binom{x}{i}\binom{(x-i)c+trash}{h-k}.$$

- $x$ represents the number of desired types we will see which we have not already seen.
- $h-k$ is the number of cards we are redrawing.
- $trash=(total-hand)-count\cdot(d-k)$ is the number of remaining cards which are not desirable.

- $\binom{x}{i}$ represents picking $i$ of these types that we will avoid when redrawing.
- $\binom{(x-i)c+trash}{h-k}$ represents the number of ways of redrawing cards that miss these $i$ types.

This is the usual inclusion-exclusion principle style formula, which is, unfortunately, not obvious. We will use this to compute our sought after $P(\text{m Unique}_f|\text{k Unique}_i)$ as. $x$ will play the role of $m-k$, the number of new desired types we will see. This gives us the formula:

$$P(\text{m Unique}_f|\text{k Unique}_i)=\frac{\binom{d-k}{m-k}\cdot Ways(m-k)}{\binom{t-h}{h-k}}.$$

- $d$ denotes the number of types of desired cards and so $d-k$ represents the number of desired types we have not yet seen.
- $\binom{d-k}{m-k}$ represents the number of ways of picking $m-k$ types from the types we have not yet seen.
- $\binom{t-h}{h-k}$ represents all of the ways we could redraw cards from the remaining deck.

In [199]:
def ways(x, unique_desired, desired_count, total_cards, hand_size, k):
    """Compute ways to draw x new unique types using inclusion-exclusion principle."""
    num_to_redraw = hand_size - k
    
    # Trash = remaining undesired cards after keeping k unique types
    # = (remaining cards in deck) - (remaining copies of unseen desired types)
    trash = (total_cards - hand_size) - desired_count * (unique_desired - k)
    
    total_sum = 0
    for i in range(x + 1):
        sign = (-1) ** i
        comb_x_i = comb(x, i, exact=True)
        
        # New desired types available = (x - i) types with desired_count copies each
        new_desired_available = desired_count * (x - i)
        total_available = new_desired_available + trash
        
        comb_available = comb(total_available, num_to_redraw, exact=True)
        
        term = sign * comb_x_i * comb_available
        total_sum += term
    
    return total_sum


def compute_p_m_given_k(m, k, unique_desired, desired_count, total_cards, hand_size):
    """Compute P(m unique final | k unique initial) conditional probability."""
    
    # Edge case: impossible situations return 0
    if m < k:
        # Can't have fewer unique types in final hand than initial
        return 0.0
    if m > unique_desired or k > unique_desired:
        # Can't have more unique types than exist
        return 0.0
    if k < 0 or m < 0:
        # Negative values are invalid
        return 0.0
    if k > hand_size or m > hand_size:
        # Can't have more unique types than hand size
        return 0.0
    
    x = m - k  # Number of new unique types to draw in redraw
    # the cases where x would invalidate the computation have already been excluded above.
    
    num_to_redraw = hand_size - k
    
    # Compute numerator: C(d-k, m-k) * Ways(m-k)
    ways_val = ways(x, unique_desired, desired_count, total_cards, hand_size, k)
    comb_new_types = comb(unique_desired - k, x, exact=True)
    numerator = comb_new_types * ways_val
    
    # Compute denominator: C(t-h, h-k)
    remaining_cards = total_cards - hand_size
    denominator = comb(remaining_cards, num_to_redraw, exact=True)
    
    if denominator == 0:
        return 0.0
    
    return numerator / denominator


def build_conditional_prob_dataframe(unique_desired, desired_count, total_cards, hand_size):
    """Build DataFrame of conditional probabilities P(m | k) for all m and k."""
    data = []
    
    for m in range(unique_desired + 1):
        row = {'m': m}
        
        for k in range(unique_desired + 1):
            prob = compute_p_m_given_k(m, k, unique_desired, desired_count, total_cards, hand_size)
            row[f'P(m|{k=})'] = prob # type: ignore
        data.append(row)
    
    return pd.DataFrame(data)


def compute_p_m_final(m, df_cond_prob, unique_desired, desired_count, num_undesired, 
                      total_cards, hand_size):
    
    """Compute P(m unique final) using law of total probability."""
    
    # Get the row for this m value from conditional probabilities
    m_row = df_cond_prob[df_cond_prob['m'] == m].iloc[0]
        # Get P(m | k) from the dataframe
    # Sum over all k: P(m|k) * P(k initial)
    total_prob = 0.0
    for k in range(unique_desired + 1):
        # Get P(m | k) from the dataframe
        p_m_given_k = m_row[f'P(m|{k=})']
        
        # Get P(k initial) using compute_pk_unique
        p_k_initial = compute_pk_unique(k, unique_desired, desired_count, num_undesired, 
                                        total_cards, hand_size)
        
        # Add contribution: P(m | k) * P(k)
        total_prob += p_m_given_k * p_k_initial
    
    return total_prob


def build_final_prob_dataframe(df_cond_prob, unique_desired, desired_count, num_undesired,
                               total_cards, hand_size):
    """Build DataFrame of final marginal probabilities P(m final) for all m values."""
    
    data = []
    
    for m in range(unique_desired + 1):
        p_m = compute_p_m_final(m, df_cond_prob, unique_desired, desired_count, 
                                num_undesired, total_cards, hand_size)
        
        data.append({
            'm': m,
            'P(m final)': p_m,
            'P(m final) %': f"{p_m * 100:.4f}%"
        })
    
    return pd.DataFrame(data)


# Example: Build the conditional probability dataframe for a specific scenario
print("Conditional Probability Table: P(m unique final | k unique initial)")
print("=" * 100)

unique_desired_test = 1
desired_count_test = 2
total_cards_test = 36
hand_size_test = 5

df_cond_prob = build_conditional_prob_dataframe(unique_desired_test, desired_count_test, 
                                                total_cards_test, hand_size_test)
print(df_cond_prob.to_string(index=False))
print("=" * 100)
print(f"\nScenario: {unique_desired_test} unique desired types with {desired_count_test} copies each")
print(f"Deck: {total_cards_test} cards total, initial hand: {hand_size_test} cards")

# Example: Compute final marginal probabilities using law of total probability
print("\n\n" + "=" * 100)
print("Final Marginal Probability Distribution: P(m unique final)")
print("=" * 100)
print("Using law of total probability: P(m) = Σ_k P(m|k) * P(k initial)")
print("=" * 100)

# Compute number of undesired cards for the test scenario
num_undesired_test = total_cards_test - (unique_desired_test * desired_count_test)

# Build the final probability dataframe
df_final_prob = build_final_prob_dataframe(df_cond_prob, unique_desired_test, 
                                           desired_count_test, num_undesired_test,
                                           total_cards_test, hand_size_test)

Conditional Probability Table: P(m unique final | k unique initial)
 m  P(m|k=0)  P(m|k=1)
 0  0.698925       0.0
 1  0.301075       1.0

Scenario: 1 unique desired types with 2 copies each
Deck: 36 cards total, initial hand: 5 cards


Final Marginal Probability Distribution: P(m unique final)
Using law of total probability: P(m) = Σ_k P(m|k) * P(k initial)


In [200]:
# Sanity Check: unique_desired=1 should match hypergeometric distribution
# Test: P(k=1) from compute_pk_unique should equal 1 - C(36-n, 5)/C(36, 5)

print("=" * 70)
print("SANITY CHECK: unique_desired=1 vs Hypergeometric Distribution")
print("=" * 70)

from scipy.special import comb

# Test parameters
total_cards = 36
hand_size = 5
unique_desired = 1

# Test for different desired_count values
test_counts = [1, 2, 3, 4, 5]

print(f"\nTesting: unique_desired={unique_desired}, total_cards={total_cards}, hand_size={hand_size}\n")
print(f"{'n (desired)':<12} {'compute_pk_unique(1)':<20} {'Hypergeometric':<20} {'Match':<10}")
print("-" * 62)

all_match = True
for desired_count in test_counts:
    num_undesired = total_cards - desired_count
    
    # Compute using our function
    prob_from_func = compute_pk_unique(
        k=1, 
        unique_desired=unique_desired, 
        desired_count=desired_count,
        num_undesired=num_undesired,
        total_cards=total_cards,
        hand_size=hand_size
    )
    
    # Compute using hypergeometric formula: P(k=1) = 1 - C(36-n, 5)/C(36, 5)
    prob_hypergeometric = 1 - (comb(total_cards - desired_count, hand_size, exact=True) / 
                                comb(total_cards, hand_size, exact=True))
    
    # Check if they match (within floating point tolerance)
    match = abs(prob_from_func - prob_hypergeometric) < 1e-10
    all_match = all_match and match
    
    print(f"{desired_count:<12} {prob_from_func:<20.15f} {prob_hypergeometric:<20.15f} {'✓' if match else '✗':<10}")

print("\n" + "=" * 70)
print(f"All tests passed: {all_match}")
print("=" * 70)

SANITY CHECK: unique_desired=1 vs Hypergeometric Distribution

Testing: unique_desired=1, total_cards=36, hand_size=5

n (desired)  compute_pk_unique(1) Hypergeometric       Match     
--------------------------------------------------------------
1            0.138888888888889    0.138888888888889    ✓         
2            0.261904761904762    0.261904761904762    ✓         
3            0.370448179271709    0.370448179271709    ✓         
4            0.465834818775995    0.465834818775995    ✓         
5            0.549298128342246    0.549298128342246    ✓         

All tests passed: True


## Draw $7$ Keep $5$
The other mulligan system is much easier to make computations about. We only need to compute the probabilities for the initial draw and then consider how the cases work after we discard $2$ cards.

### Initial Draw
We now compute $P(\text{k Unique})$ for various values of $k$ when we draw $7$ cards. We can reuse the functions we developed in the previous case.

In [201]:
# Compute P(k unique) for draw 7, keep 5 system
# Using the same functions developed for the hard mulligan analysis

hand_size = 7
total_cards = 36
unique_desired = 3
desired_count = 2
num_undesired = total_cards - (unique_desired * desired_count)


print("=" * 80)
print(f"Draw {hand_size}, Keep 5 - Initial Draw Analysis")
print(f"Scenario: {unique_desired} unique desired types with {desired_count} copies each")
print(f"Deck: {total_cards} cards total ({unique_desired * desired_count} desired, {num_undesired} undesired)")
print("=" * 80)

# Compute P(k unique) distribution for initial draw
print(f"\nP(k unique desired in initial hand of {hand_size}):")
print("-" * 80)

data = []
for k in range(unique_desired + 1):
    p_k = compute_pk_unique(
        k=k,
        unique_desired=unique_desired,
        desired_count=desired_count,
        num_undesired=num_undesired,
        total_cards=total_cards,
        hand_size=hand_size
    )
    
    data.append({
        'k (unique desired)': k,
        'P(k)': p_k,
        'P(k) %': f"{p_k * 100:.4f}%"
    })

df_pk_draw7 = pd.DataFrame(data)
print(df_pk_draw7.to_string(index=False))
print("-" * 80)
print(f"Verification: Sum of P(k) = {df_pk_draw7['P(k)'].sum():.10f} (should be 1.0)")
print("=" * 80)

Draw 7, Keep 5 - Initial Draw Analysis
Scenario: 3 unique desired types with 2 copies each
Deck: 36 cards total (6 desired, 30 undesired)

P(k unique desired in initial hand of 7):
--------------------------------------------------------------------------------
 k (unique desired)     P(k)   P(k) %
                  0 0.243876 24.3876%
                  1 0.477997 47.7997%
                  2 0.245710 24.5710%
                  3 0.032416  3.2416%
--------------------------------------------------------------------------------
Verification: Sum of P(k) = 1.0000000000 (should be 1.0)


#### Additional Scenarios
We compute P(k unique) for multiple scenarios when drawing 7 cards, using the same scenarios as the friendliest mulligan analysis. This allows comparison of expected values across both mulligan systems.


In [202]:
# Set hand_size for Draw 7 Keep 5 analysis
hand_size_draw7 = 7

df_pk_draw7 = build_pk_unique_dataframe_no_expectations(scenarios, total_cards, hand_size_draw7)
df_complete_draw7 = df_pk_draw7.copy()
df_complete_draw7 = adjoin_expected_values(df_complete_draw7, hand_size_param=hand_size_draw7)

print("P(k unique) Distributions for Each Scenario - Draw 7 Keep 5 (Initial Draw)")
print("=" * 100)
print(df_complete_draw7.to_string(index=False))
print("=" * 100)



P(k unique) Distributions for Each Scenario - Draw 7 Keep 5 (Initial Draw)
 unique_desired  desired_count   P(k=0)   P(k=1)   P(k=2)   P(k=3)   P(k=4)   P(k=5)   P(k=6)  E[desired]  E[unique]
              1              2 0.644444 0.355556 0.000000 0.000000 0.000000 0.000000 0.000000    0.388889   0.355556
              2              2 0.403209 0.482472 0.114320 0.000000 0.000000 0.000000 0.000000    0.777778   0.711111
              3              2 0.243876 0.477997 0.245710 0.032416 0.000000 0.000000 0.000000    1.166667   1.066667
              4              2 0.141841 0.408142 0.343781 0.098426 0.007810 0.000000 0.000000    1.555556   1.422222
              5              2 0.078800 0.315201 0.389953 0.183016 0.031525 0.001505 0.000000    1.944444   1.777778
              4              3 0.041461 0.259677 0.435458 0.231266 0.032137 0.000000 0.000000    2.333333   1.952941
              6              1 0.243876 0.426783 0.256070 0.065659 0.007295 0.000313 0.000004    1.166667 

### $P(\text{m Unique}_f)$
These final probabilities are easy to compute since we do not redraw any cards with this mulligan. This means that we have
$$P(\text{m Unique}_f)=P(\text{m Unique}_i)$$
when $m$ is $5$ or less. So no new information is gained.

## Comparison

In [203]:

def build_draw7_final_distribution(unique_desired, desired_count):
    num_undesired = total_cards - (unique_desired * desired_count)
    
    final_data = []
    
    for m in range(unique_desired + 1):
        prob_m_final = compute_pk_unique(m, unique_desired, desired_count, num_undesired, 
                                           total_cards, hand_size_draw7)
        final_data.append({
            'm': m,
            'P(m final)': prob_m_final,
            'P(m final) %': f"{prob_m_final * 100:.4f}%"
        })
    
    return pd.DataFrame(final_data)


def build_scenario_comparison_dataframes(scenario):
    
    unique_desired = scenario['unique_desired']
    desired_count = scenario['desired_count']
    num_undesired = total_cards - (unique_desired * desired_count)
    
    # ========== FRIENDLIEST MULLIGAN (hand size 5) ==========
    df_cond_friendly = build_conditional_prob_dataframe(unique_desired, desired_count, total_cards, hand_size)
    df_final_friendly = build_final_prob_dataframe(df_cond_friendly, unique_desired, desired_count, num_undesired, total_cards, hand_size)
    
    # ========== DRAW 7 KEEP 5 (NO REDRAW) ==========
    df_final_draw7 = build_draw7_final_distribution(unique_desired, desired_count)
    
    return {
        'df_final_friendly': df_final_friendly,
        'df_final_draw7': df_final_draw7,
        'unique_desired': unique_desired,
        'desired_count': desired_count
    }


def compare_mulligan_strategies(scenario):
    
    # Build dataframes for this scenario
    dfs = build_scenario_comparison_dataframes(scenario)
    df_final_friendly = dfs['df_final_friendly']
    df_final_draw7 = dfs['df_final_draw7']
    
    # Extract probabilities and compute expected values
    final_probs_friendly = dict(zip(df_final_friendly['m'], df_final_friendly['P(m final)']))
    ev_friendly = sum(m * final_probs_friendly[m] for m in final_probs_friendly.keys())
    
    final_probs_draw7 = dict(zip(df_final_draw7['m'], df_final_draw7['P(m final)']))
    ev_draw7 = sum(m * final_probs_draw7[m] for m in final_probs_draw7.keys())
    
    # Build comparison table
    comparison_data = []
    max_m_combined = max(max(final_probs_friendly.keys()), max(final_probs_draw7.keys()))
    
    for m in range(max_m_combined + 1):
        p_friendly = final_probs_friendly.get(m, 0.0)
        p_draw7 = final_probs_draw7.get(m, 0.0)
        diff = p_friendly - p_draw7
        
        comparison_data.append({
            'm': m,
            'Friendliest %': format_percent(p_friendly),
            'Draw 7 Keep 5 %': format_percent(p_draw7),
            'Difference %': format_percent(diff) if abs(diff) > 1e-10 else '—'
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    
    return {
        'df_final_friendly': df_final_friendly,
        'df_final_draw7': df_final_draw7,
        'friendly_dist': final_probs_friendly,
        'draw7_dist': final_probs_draw7,
        'friendly_ev': ev_friendly,
        'draw7_ev': ev_draw7,
        'comparison_df': df_comparison
    }


def print_scenario_comparison(scenario, result):
   
    unique_desired = scenario['unique_desired']
    desired_count = scenario['desired_count']
    
    print(f"\n\nScenario: {unique_desired} unique desired types with {desired_count} copies each")
    print("=" * 120)
    
    # Print Friendliest Mulligan results
    print(f"\n1. FRIENDLIEST MULLIGAN (hand size 5, redraw available)")
    print("-" * 120)
    print(result['df_final_friendly'].to_string(index=False))
    print(f"Expected unique cards: {result['friendly_ev']:.6f}")
    
    # Print Draw 7 Keep 5 results
    print(f"\n2. DRAW 7 KEEP 5 (hand size 7, no redraw)")
    print("-" * 120)
    print(result['df_final_draw7'].to_string(index=False))
    print(f"Expected unique cards: {result['draw7_ev']:.6f}")
    
    # Print comparison table
    print(f"\n3. DIRECT COMPARISON")
    print("-" * 120)
    print(result['comparison_df'].to_string(index=False))
    
    # Print analysis
    ev_diff = result['friendly_ev'] - result['draw7_ev']
    print(f"\nExpected Values:")
    print(f"  Friendliest Mulligan: {result['friendly_ev']:.6f} unique cards")
    print(f"  Draw 7 Keep 5:        {result['draw7_ev']:.6f} unique cards")
    print(f"  Difference:           {ev_diff:+.6f}")
    
    if abs(ev_diff) < 1e-6:
        print(f"  → Strategies are EQUIVALENT for this scenario")
    elif ev_diff < 0:
        print(f"  → Draw 7 Keep 5 is BETTER (provides {abs(ev_diff):.6f} more unique cards)")
    else:
        print(f"  → Friendliest Mulligan is BETTER (provides {ev_diff:.6f} more unique cards)")


In [204]:
# Compare both strategies across all scenarios using the refactored functions

print("\n" + "=" * 120)
print("COMPARISON: Friendliest Mulligan vs Draw 7 Keep 5")
print("Both optimizing for UNIQUE DESIRED CARDS in final hand")
print("=" * 120)

# Compute results for all scenarios
comparison_results = {}

for scenario in scenarios:
    result = compare_mulligan_strategies(scenario)
    comparison_results[f"{scenario['unique_desired']}_{scenario['desired_count']}"] = result
    print_scenario_comparison(scenario, result)

# Summary table
print("\n\n" + "=" * 120)
print("SUMMARY: Expected Unique Desired Cards in Final Hand")
print("=" * 120)
print(f"{'Scenario':<25} {'Friendliest':<20} {'Draw 7 Keep 5':<20} {'Difference':<20}")
print("-" * 120)

for scenario in scenarios:
    key = f"{scenario['unique_desired']}_{scenario['desired_count']}"
    result = comparison_results[key]
    friendly_ev = result['friendly_ev']
    draw7_ev = result['draw7_ev']
    diff = friendly_ev - draw7_ev
    
    print(f"{scenario['unique_desired']} unique × {scenario['desired_count']} copies  <{key:<18}  {friendly_ev:<20.6f} {draw7_ev:<20.6f} {diff:<+20.6f}")

print("=" * 120)


COMPARISON: Friendliest Mulligan vs Draw 7 Keep 5
Both optimizing for UNIQUE DESIRED CARDS in final hand


Scenario: 1 unique desired types with 2 copies each

1. FRIENDLIEST MULLIGAN (hand size 5, redraw available)
------------------------------------------------------------------------------------------------------------------------
 m  P(m final) P(m final) %
 0    0.366667     36.6667%
 1    0.633333     63.3333%
Expected unique cards: 0.633333

2. DRAW 7 KEEP 5 (hand size 7, no redraw)
------------------------------------------------------------------------------------------------------------------------
 m  P(m final) P(m final) %
 0    0.644444     64.4444%
 1    0.355556     35.5556%
Expected unique cards: 0.355556

3. DIRECT COMPARISON
------------------------------------------------------------------------------------------------------------------------
 m Friendliest % Draw 7 Keep 5 % Difference %
 0      36.6667%        64.4444%    -27.7778%
 1      63.3333%        35.5556

In [205]:
# Detailed Probability Distribution Comparison

print("\n\n" + "=" * 120)
print("DETAILED PROBABILITY DISTRIBUTION COMPARISON")
print("=" * 120)

for scenario in scenarios:
    key = f"{scenario['unique_desired']}_{scenario['desired_count']}"
    result = comparison_results[key]
    
    unique_desired = scenario['unique_desired']
    desired_count = scenario['desired_count']
    
    print(f"\n\nScenario: {unique_desired} unique types × {desired_count} copies")
    print("-" * 120)
    
    # Print comparison table
    print(result['comparison_df'].to_string(index=False))
    
    # Analysis
    friendly_ev = result['friendly_ev']
    draw7_ev = result['draw7_ev']
    ev_diff = friendly_ev - draw7_ev
    
    print(f"\nExpected Values:")
    print(f"  Friendliest Mulligan: {friendly_ev:.6f} unique cards")
    print(f"  Draw 7 Keep 5:        {draw7_ev:.6f} unique cards")
    print(f"  Difference:           {ev_diff:+.6f}")
    
    if abs(ev_diff) < 1e-6:
        print(f"  → Strategies are EQUIVALENT for this scenario")
    elif ev_diff < 0:
        print(f"  → Draw 7 Keep 5 is BETTER (provides {abs(ev_diff):.6f} more unique cards)")
    else:
        print(f"  → Friendliest Mulligan is BETTER (provides {ev_diff:.6f} more unique cards)")

print("\n\n" + "=" * 120)
print("OVERALL SUMMARY")
print("=" * 120)

summary_table = []
for scenario in scenarios:
    key = f"{scenario['unique_desired']}_{scenario['desired_count']}"
    result = comparison_results[key]
    friendly_ev = result['friendly_ev']
    draw7_ev = result['draw7_ev']
    better = "Friendliest" if friendly_ev > draw7_ev else "Draw 7" if draw7_ev > friendly_ev else "Tie"
    summary_table.append({
        'unique_desired': scenario['unique_desired'],
        'desired_count': scenario['desired_count'],
        'Friendliest': f"{friendly_ev:.6f}",
        'Draw 7 Keep 5': f"{draw7_ev:.6f}",
        'Better Strategy': better
    })

df_summary_comp = pd.DataFrame(summary_table)
print(df_summary_comp.to_string(index=False))
print("=" * 120)



DETAILED PROBABILITY DISTRIBUTION COMPARISON


Scenario: 1 unique types × 2 copies
------------------------------------------------------------------------------------------------------------------------
 m Friendliest % Draw 7 Keep 5 % Difference %
 0      36.6667%        64.4444%    -27.7778%
 1      63.3333%        35.5556%     27.7778%

Expected Values:
  Friendliest Mulligan: 0.633333 unique cards
  Draw 7 Keep 5:        0.355556 unique cards
  Difference:           +0.277778
  → Friendliest Mulligan is BETTER (provides 0.277778 more unique cards)


Scenario: 2 unique types × 2 copies
------------------------------------------------------------------------------------------------------------------------
 m Friendliest % Draw 7 Keep 5 % Difference %
 0      12.4183%        40.3209%    -27.9026%
 1      51.1111%        48.2472%      2.8639%
 2      36.4706%        11.4320%     25.0386%

Expected Values:
  Friendliest Mulligan: 1.240523 unique cards
  Draw 7 Keep 5:        0.711111

Unsurprisingly, the "Friendliest" mulligan is preferable with this strategy.